# Parte 1 – Scraping de decretos de emergencia

In [1]:
import time, re
import requests
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

#### Paso 1 – robots.txt de gob.pe

In [50]:
headers = {"User-Agent": "Mozilla/5.0"}

robots = requests.get("https://www.gob.pe/robots.txt", headers=headers, timeout=30)
print(robots.text)

# See http://www.robotstxt.org/robotstxt.html for documentation on how to use the robots.txt file
#
# To ban all spiders from the entire site uncomment the next two lines:
User-agent: *
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: AhrefsBot
Disallow: /

User-agent: GPTBot
Crawl-delay: 5
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: OAI-SearchBot
Crawl-delay: 2
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

Sitemap: https://www.gob.pe/sitemaps/sitemap.xml.gz



##### Análisis del robots.txt de gob.pe
**¿Qué prohíbe?** Para cualquier bot (`User-agent: *`), prohíbe acceder a la sección `/admin/` y `sheet=`. Además, bloquea por completo el sitio para `AhrefsBot`.

**¿Está permitida la página de búsquedas?** La ruta `/busquedas` no aparece en ninguna regla `Disallow`, por lo que se conluye que esta permitida.

**¿Por qué igual conviene hacer pausas?** Aunque no se nos exige un `Crawl-delay`, conviene hacer pausas para: 1. evitar que nos bloqueen y 2. Saturar el servicio que brinda gob.pe, porque ademas afectaria a los ciudadanos. Por otro lado, la pagina sí le pide a otros bots `Crawl-delay`: 5 segundos a `GPTBot` y 2 segundos a `OAI-SearchBot`.

#### Paso 2 – Abrir con Selenium la búsqueda de un mes

In [29]:
BASE = ("https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm"
        "&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent")

url_mes = BASE + "&desde=01-01-2024&hasta=31-01-2024"

navegador = webdriver.Chrome()
navegador.get(url_mes)

WebDriverWait(navegador, 30).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "article")))
time.sleep(3)  # margen extra: la página sigue dibujando resultados
print("Contenido cargado")
print("Artículos encontrados:", len(navegador.find_elements(By.TAG_NAME, "article")))

Contenido cargado
Artículos encontrados: 11


#### Paso 3 – Leer el total de resultados ("N Resultados")

In [30]:
# el total aparece arriba de la lista como "N Resultados", dentro de <h2 role="status"><span>
contador = WebDriverWait(navegador, 30).until(
    EC.visibility_of_element_located((By.CSS_SELECTOR, "h2[role='status'] span")))
texto_total = contador.text                             # ej. "11 Resultados"
total_resultados = int(re.sub(r"\D", "", texto_total))  # nos quedamos solo con los dígitos

print("Texto en la página:", texto_total)
print("Total de resultados:", total_resultados)

Texto en la página: 11 Resultados
Total de resultados: 11


#### Paso 4 – Extraer los datos de cada resultado

In [31]:
def extraer_articulo(art):
    # enlace principal: trae el número de la norma y su URL
    link = art.find_element(By.CSS_SELECTOR, "a.link-principal")

    # la fecha es la línea que empieza con "Publicado:"
    fecha = ""
    for linea in art.text.split("\n"):
        if linea.startswith("Publicado:"):
            fecha = linea.replace("Publicado:", "").strip()

    # el título es el párrafo debajo de la fecha (viene cortado con "...")
    titulo = art.find_element(By.TAG_NAME, "p").text

    return {
        "numero": link.text.strip(),
        "fecha": fecha,
        "titulo": titulo.strip(),
        "enlace": link.get_attribute("href"),
    }

In [32]:
# prueba con los resultados de enero que ya están abiertos en el navegador
articulos = navegador.find_elements(By.TAG_NAME, "article")
filas_enero = []
for art in articulos:
    filas_enero.append(extraer_articulo(art))

print("Normas extraídas:", len(filas_enero))
pd.DataFrame(filas_enero)

Normas extraídas: 11


,numero,fecha,titulo,enlace
0,Decreto Supremo N.° 010-2024-PCM,30 de enero de 2024,Prorrogar por el termino de sesenta (60) días ...,https://www.gob.pe/institucion/pcm/normas-lega...
1,Decreto Supremo N.° 009-2024-PCM,29 de enero de 2024,Prorrogar por el termino de sesenta (60) días ...,https://www.gob.pe/institucion/pcm/normas-lega...
2,Decreto Supremo N.° 008-2024-PCM,29 de enero de 2024,Declarar por el termino de treinta (30) días c...,https://www.gob.pe/institucion/pcm/normas-lega...
3,Decreto Supremo N.° 007-2024-PCM,29 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
4,Decreto Supremo N.° 006-2024-PCM,29 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
5,Decreto Supremo N.° 005-2024-PCM,29 de enero de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
6,Decreto Supremo N.° 004-2024-PCM,26 de enero de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
7,Resolución N.° 0214-2024-TCE-S6,18 de enero de 2024,N° 10304103618); por su responsabilidad al hab...,https://www.gob.pe/institucion/oece/normas-leg...
8,Decreto Supremo N.° 003-2024-PCM,10 de enero de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
9,Decreto Supremo N.° 002-2024-PCM,9 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...


#### Paso 5 – Recorrer los meses de la temporada (enero a mayo 2024)

In [33]:
def scrapear_mes(navegador, desde, hasta):
    navegador.get(BASE + f"&desde={desde}&hasta={hasta}")

    # esperar al total o, si no hay resultados, al aviso "No se encontraron resultados"
    WebDriverWait(navegador, 30).until(EC.any_of(
        EC.visibility_of_element_located((By.CSS_SELECTOR, "h2[role='status'] span")),
        EC.visibility_of_element_located((By.XPATH, "//h2[contains(text(), 'No se encontraron resultados')]"))
    ))
    time.sleep(3)  # margen extra: la página sigue dibujando resultados

    # sin resultados la página no muestra el contador
    contador = navegador.find_elements(By.CSS_SELECTOR, "h2[role='status'] span")
    if len(contador) == 0:
        return 0, []
    total = int(re.sub(r"\D", "", contador[0].text))

    filas = []
    for art in navegador.find_elements(By.TAG_NAME, "article"):
        filas.append(extraer_articulo(art))
    return total, filas

In [34]:
# prueba: el 1 de enero de 2024 no tiene resultados
print(scrapear_mes(navegador, "01-01-2024", "01-01-2024"))
time.sleep(2)

(0, [])


In [36]:
# meses de la temporada de lluvias (2024 es bisiesto: febrero termina el 29)
meses = [
    ("enero",   "01-01-2024", "31-01-2024"),
    ("febrero", "01-02-2024", "29-02-2024"),
    ("marzo",   "01-03-2024", "31-03-2024"),
    ("abril",   "01-04-2024", "30-04-2024"),
    ("mayo",    "01-05-2024", "31-05-2024"),
]

todas = []
control = []

for mes, desde, hasta in meses:
    total, filas = scrapear_mes(navegador, desde, hasta)
    for fila in filas:
        todas.append(fila)
    control.append({"mes": mes,
                    "resultados_totales": total,
                    "resultados_extraidos": len(filas)})
    print(f"{mes}: {total} resultados, {len(filas)} extraídos")
    time.sleep(2)  # pausa entre meses para no saturar gob.pe

navegador.quit()
print("Navegador cerrado")
print("Normas extraídas en la temporada:", len(todas))

enero: 11 resultados, 11 extraídos
febrero: 11 resultados, 11 extraídos
marzo: 11 resultados, 11 extraídos
abril: 15 resultados, 15 extraídos
mayo: 3 resultados, 3 extraídos
Navegador cerrado
Normas extraídas en la temporada: 51


#### Paso 6 – Verificación

In [37]:
# control por mes: el total de la página debe coincidir con lo extraído
df_control = pd.DataFrame(control)
df_control["coincide"] = df_control["resultados_totales"] == df_control["resultados_extraidos"]
print("¿Coinciden todos los meses?:", df_control["coincide"].all())
df_control

¿Coinciden todos los meses?: True


,mes,resultados_totales,resultados_extraidos,coincide
0,enero,11,11,True
1,febrero,11,11,True
2,marzo,11,11,True
3,abril,15,15,True
4,mayo,3,3,True


#### Paso 7 – Eliminar normas repetidas

In [56]:
normas = pd.DataFrame(todas) #convierte la lista de resultados en una tabla
antes = len(normas) # se guardan la cantidad de filas antes de limpiar para hacer la comparación
normas = normas.drop_duplicates(subset="enlace").reset_index(drop=True) #drop_duplicates: elimina las filas cuyo enlace ya apareció antes y conserva la primera. 
#con reset_index: renumera las filas de 0 en adelante, para que no queden huecos en el índice.
print(f"Filas antes: {antes} | después: {len(normas)} | repetidas eliminadas: {antes - len(normas)}")  
# muestra el resultado: 51 antes, 51 después, 0 repetidas.


Filas antes: 51 | después: 51 | repetidas eliminadas: 0


#### 8. Quedarnos solo con normas de la PCM

In [57]:
normas.head(10) # En la Fila 7 se tiene la Resolución del OECE (enlace /institucion/oece/), no es de la PCM y se debe eliminar.

,numero,fecha,titulo,enlace
0,Decreto Supremo N.° 010-2024-PCM,30 de enero de 2024,Prorrogar por el termino de sesenta (60) días ...,https://www.gob.pe/institucion/pcm/normas-lega...
1,Decreto Supremo N.° 009-2024-PCM,29 de enero de 2024,Prorrogar por el termino de sesenta (60) días ...,https://www.gob.pe/institucion/pcm/normas-lega...
2,Decreto Supremo N.° 008-2024-PCM,29 de enero de 2024,Declarar por el termino de treinta (30) días c...,https://www.gob.pe/institucion/pcm/normas-lega...
3,Decreto Supremo N.° 007-2024-PCM,29 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
4,Decreto Supremo N.° 006-2024-PCM,29 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
5,Decreto Supremo N.° 005-2024-PCM,29 de enero de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
6,Decreto Supremo N.° 004-2024-PCM,26 de enero de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
7,Resolución N.° 0214-2024-TCE-S6,18 de enero de 2024,N° 10304103618); por su responsabilidad al hab...,https://www.gob.pe/institucion/oece/normas-leg...
8,Decreto Supremo N.° 003-2024-PCM,10 de enero de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
9,Decreto Supremo N.° 002-2024-PCM,9 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...


In [58]:
# 8.1. Creamos una columna de Verdadero/Falso: True si el enlace pertenece a la PCM.
#    regex=False hace que se busque el texto literal "/institucion/pcm/", sin tratarlo como patrón.
#Aqui vemos que la fila 7 es False
es_pcm = normas["enlace"].str.contains("/institucion/pcm/", regex=False)
es_pcm.head(10)

0     True
1     True
2     True
3     True
4     True
5     True
6     True
7    False
8     True
9     True
Name: enlace, dtype: bool

In [59]:
# 8.2. El símbolo ~ invierte los valores (True pasa a False y al revés), así que ~es_pcm marca
#    las normas que NO son de la PCM. .sum() cuenta los True: 8 normas de otras instituciones.
print("Normas de otras instituciones que quitamos:", (~es_pcm).sum())

Normas de otras instituciones que quitamos: 8


In [60]:
# 8.3. Antes de borrarlas, las mostramos: así comprobamos qué estamos eliminando.
print(normas.loc[~es_pcm, ["numero", "enlace"]].to_string(index=False))

                         numero                                                                      enlace
Resolución N.° 0214-2024-TCE-S6 https://www.gob.pe/institucion/oece/normas-legales/5187307-0214-2024-tce-s6
Resolución N.° 0634-2024-TCE-S5 https://www.gob.pe/institucion/oece/normas-legales/5260816-0634-2024-tce-s5
Resolución N.° 0533-2024-TCE-S4 https://www.gob.pe/institucion/oece/normas-legales/5197999-0533-2024-tce-s4
Resolución N.° 0506-2024-TCE-S5 https://www.gob.pe/institucion/oece/normas-legales/5193883-0506-2024-tce-s5
Resolución N.° 0437-2024-TCE-S3 https://www.gob.pe/institucion/oece/normas-legales/5192380-0437-2024-tce-s3
Resolución N.° 0991-2024-TCE-S6 https://www.gob.pe/institucion/oece/normas-legales/5421041-0991-2024-tce-s6
Resolución N.° 0972-2024-TCE-S2 https://www.gob.pe/institucion/oece/normas-legales/5420398-0972-2024-tce-s2
Resolución N.° 1071-2024-TCE-S4 https://www.gob.pe/institucion/oece/normas-legales/5464861-1071-2024-tce-s4


In [61]:
# 8.4. Nos quedamos solo con las filas donde es_pcm es True y renumeramos el índice desde 0.
normas = normas[es_pcm].reset_index(drop=True)
normas.head(10)

,numero,fecha,titulo,enlace
0,Decreto Supremo N.° 010-2024-PCM,30 de enero de 2024,Prorrogar por el termino de sesenta (60) días ...,https://www.gob.pe/institucion/pcm/normas-lega...
1,Decreto Supremo N.° 009-2024-PCM,29 de enero de 2024,Prorrogar por el termino de sesenta (60) días ...,https://www.gob.pe/institucion/pcm/normas-lega...
2,Decreto Supremo N.° 008-2024-PCM,29 de enero de 2024,Declarar por el termino de treinta (30) días c...,https://www.gob.pe/institucion/pcm/normas-lega...
3,Decreto Supremo N.° 007-2024-PCM,29 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
4,Decreto Supremo N.° 006-2024-PCM,29 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
5,Decreto Supremo N.° 005-2024-PCM,29 de enero de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
6,Decreto Supremo N.° 004-2024-PCM,26 de enero de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
7,Decreto Supremo N.° 003-2024-PCM,10 de enero de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
8,Decreto Supremo N.° 002-2024-PCM,9 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
9,Decreto Supremo N.° 001-2024-PCM,4 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...


In [62]:
# 5. Control final: Teniamos 51 normas de las cuales se detectó que 8 no eran de la PCM por lo que debn quedar 43 normas solo de la PCM.
print("\nNormas solo de la PCM:", len(normas))


Normas solo de la PCM: 43


#### 9. Título completo con requests + BeautifulSoup

In [63]:
import pandas as pd
import requests
from bs4 import BeautifulSoup


In [64]:
def titulo_completo(enlace):
    # Pedimos la página de la norma. No necesita JavaScript, así que basta requests (no Selenium).
    # headers lleva el User-Agent porque gob.pe rechaza pedidos que no se identifican como navegador.
    respuesta = requests.get(enlace, headers=headers, timeout=30)

    # Si la página respondió con error (404, 429...), avisa aquí y no más adelante con datos vacíos.
    respuesta.raise_for_status()

    # Convertimos el HTML en un objeto que podemos recorrer.
    sopa = BeautifulSoup(respuesta.text, "html.parser")

    # El título completo está dentro de <div class="description">.
    descripcion = sopa.find("div", class_="description")

    # Dentro de ese div está también la tarjeta del PDF ("DS N° 021-2024-PCM PDF 1.4 MB Descargar").
    # .decompose() la borra del HTML, para que su texto no se mezcle con el título.
    for tarjeta in descripcion.find_all("div", class_="institution-document__files"):
        tarjeta.decompose()

    # Sacamos solo el texto: " " une los fragmentos con un espacio y strip=True quita espacios sobrantes.
    return descripcion.get_text(" ", strip=True)



In [65]:
titulos = []
for enlace in normas["enlace"]:
    titulos.append(titulo_completo(enlace))
    time.sleep(1)   # 1 segundo entre normas, como pide el enunciado, para no saturar el servidor

normas["titulo_corto"] = normas["titulo"]   # guardamos el título de la lista ("...") antes de reemplazarlo

# Reemplazamos el título cortado por el completo.
normas["titulo"] = titulos

# Comprobaciones: si la limpieza funcionó, ningún título debe terminar en "..."
# ni contener "Descargar" (señal de que quedó pegado el texto de la tarjeta del PDF).
print("Títulos que siguen cortados:", normas["titulo"].str.endswith("...").sum())
print("Títulos con 'Descargar':", normas["titulo"].str.contains("Descargar").sum())

# Comparamos el título cortado con el completo en las 3 primeras normas.
normas[["numero", "titulo_corto", "titulo"]].head(3)

Títulos que siguen cortados: 0
Títulos con 'Descargar': 0


,numero,titulo_corto,titulo
0,Decreto Supremo N.° 010-2024-PCM,Prorrogar por el termino de sesenta (60) días ...,Prorrogar por el termino de sesenta (60) días ...
1,Decreto Supremo N.° 009-2024-PCM,Prorrogar por el termino de sesenta (60) días ...,Prorrogar por el termino de sesenta (60) días ...
2,Decreto Supremo N.° 008-2024-PCM,Declarar por el termino de treinta (30) días c...,Declarar por el termino de treinta (30) días c...


#### 10. Clasificar las normas por tema, tipo y motivo

In [66]:
# 1. ¿La norma trata de lluvias?
def es_lluvia(titulo):
    t = titulo.lower()   # pasamos a minúsculas para que "Lluvias" y "lluvias" se encuentren igual
    # "lluvia" también encuentra "lluvias"; devuelve True si aparece cualquiera de las dos palabras
    return "lluvia" in t or "precipitaciones" in t

# 2. ¿La norma declara o prorroga la emergencia? 
def tipo(titulo):
    t = titulo.lower()
    # Preguntamos PRIMERO por "prórroga": las prórrogas dicen "...el Estado de Emergencia
    # declarado mediante...", así que si preguntáramos primero por "declara",
    # todas las prórrogas se clasificarían mal como declaratorias.
    # Buscamos con y sin tilde por si el título la omite; "prorroga" también encuentra "Prorrogar".
    if "prórroga" in t or "prorroga" in t:
        return "prorroga"
    elif "declara" in t:
        return "declara"
    else:
        return "otro"      # ni declara ni prorroga (por ejemplo, normas que modifican un decreto)

# 3. ¿Por qué motivo se declara la emergencia?
def motivo(titulo):
    t = titulo.lower()
    if "peligro inminente" in t:
        return "peligro inminente"   # antes del daño: medida preventiva
    elif "impacto de daños" in t:
        return "impacto de daños"    # después del daño: medida de respuesta
    else:
        return "otro"                # normas sin ninguna de las dos frases exactas

In [67]:
# Aplicamos cada función a la columna "titulo" (el título completo del paso 9):
# .apply recorre fila por fila y guarda el resultado en una columna nueva.
normas["es_lluvia"] = normas["titulo"].apply(es_lluvia)
normas["tipo"] = normas["titulo"].apply(tipo)
normas["motivo"] = normas["titulo"].apply(motivo)

# Control 1: cuántas normas son de lluvias (True) y cuántas no (False)
print(normas["es_lluvia"].value_counts(), "\n")

# Control 2: tabla cruzada de tipo × motivo; margins=True agrega los totales ("All").
# Ojo: incluye también las normas que no son de lluvias, que siempre quedan con motivo "otro".
print(pd.crosstab(normas["tipo"], normas["motivo"], margins=True))

es_lluvia
False    23
True     20
Name: count, dtype: int64 

motivo    impacto de daños  otro  peligro inminente  All
tipo                                                    
declara                 12     5                  0   17
prorroga                 8    16                  2   26
All                     20    21                  2   43


In [68]:
#Revisamos las filas que quedaron como "otro" en tipo o en motivo:
otros = normas[(normas["tipo"] == "otro") | (normas["motivo"] == "otro")]
for _, fila in otros.iterrows():
    print(f"[{fila['numero']}] es_lluvia={fila['es_lluvia']} | tipo={fila['tipo']} | motivo={fila['motivo']}")
    print("   ", fila["titulo"], "\n")

[Decreto Supremo N.° 010-2024-PCM] es_lluvia=False | tipo=prorroga | motivo=otro
    Prorrogar por el termino de sesenta (60) días calendario a partir del 6 de febrero al 5 de abril de 2024, el Estado de Emergencia en los distritos que se indican en el Anexo que forma parte integrante del presente Decreto Supremo. 

[Decreto Supremo N.° 009-2024-PCM] es_lluvia=False | tipo=prorroga | motivo=otro
    Prorrogar por el termino de sesenta (60) días calendario, a partir del 1 de febrero de 2024, el Estado de Emergencia declarado en los distritos de Tambopata, Inambari, Las Piedras y Laberinto de la provincia de Tambopata y en los distritos de Madre de Dios y Huepetuhe de la provincia de Manu del departamento de Madre de Dios. La Policía Nacional del Perú mantiene el control del orden interno, con el apoyo de las Fuerzas Armadas. 

[Decreto Supremo N.° 008-2024-PCM] es_lluvia=False | tipo=declara | motivo=otro
    Declarar por el termino de treinta (30) días calendario, a partir del 5 de feb

In [69]:
#También revisamos al revés: normas que no quedaron como lluvia pero sí tienen un motivo ("peligro inminente" o "impacto de daños"), por si el filtro de lluvia se nos escapó alguna:
sospechosas = normas[~normas["es_lluvia"] & (normas["motivo"] != "otro")]
for _, fila in sospechosas.iterrows():
    print(f"[{fila['numero']}] tipo={fila['tipo']} | motivo={fila['motivo']}")
    print("   ", fila["titulo"], "\n")

[Decreto Supremo N.° 052-2024-PCM] tipo=declara | motivo=impacto de daños
    Decreto Supremo que declara el Estado de Emergencia en los distritos Pachacamac y Villa María del Triunfo de la provincia de Lima del departamento de Lima, por impacto de daños a consecuencia de explosión en estación de carga de gas natural comprimido 

[Decreto Supremo N.° 050-2024-PCM] tipo=declara | motivo=impacto de daños
    Decreto Supremo que declara el Estado de Emergencia en varios distritos de la provincia de Quispicanchi del departamento de Cusco, por impacto de daños ante la ocurrencia de movimiento sísmico 



#### 11. Solo normas de lluvias

In [70]:
lluvias = normas[normas["es_lluvia"]].reset_index(drop=True)
print("Normas de lluvias:", len(lluvias))
lluvias[["numero", "fecha", "tipo", "motivo", "titulo"]].head(10)

Normas de lluvias: 20


,numero,fecha,tipo,motivo,titulo
0,Decreto Supremo N.° 007-2024-PCM,29 de enero de 2024,prorroga,impacto de daños,Decreto Supremo que prorroga el Estado de Emer...
1,Decreto Supremo N.° 006-2024-PCM,29 de enero de 2024,prorroga,peligro inminente,Decreto Supremo que prorroga el Estado de Emer...
2,Decreto Supremo N.° 005-2024-PCM,29 de enero de 2024,declara,impacto de daños,Decreto Supremo que declara el Estado de Emerg...
3,Decreto Supremo N.° 004-2024-PCM,26 de enero de 2024,declara,impacto de daños,Decreto Supremo que declara el Estado de Emerg...
4,Decreto Supremo N.° 021-2024-PCM,28 de febrero de 2024,declara,impacto de daños,Decreto Supremo que declara el Estado de Emerg...
5,Decreto Supremo N.° 020-2024-PCM,26 de febrero de 2024,declara,impacto de daños,Decreto Supremo que declara el Estado de Emerg...
6,Decreto Supremo N.° 016-2024-PCM,9 de febrero de 2024,declara,impacto de daños,Decreto Supremo que declara el Estado de Emerg...
7,Decreto Supremo N.° 0012-2024-PCM,1 de febrero de 2024,declara,impacto de daños,Declarar el Estado de Emergencia en varios dis...
8,Decreto Supremo N.° 033-2024-PCM,26 de marzo de 2024,prorroga,impacto de daños,Decreto Supremo que prorroga el Estado de Emer...
9,Decreto Supremo N.° 032-2024-PCM,26 de marzo de 2024,prorroga,impacto de daños,Decreto Supremo que prorroga el Estado de Emer...


#### 12. Mencionar qué departamentos menciona cada decreto

In [ ]:
#Se detecta los departamentos
import re
import unicodedata

# Lista oficial de los 25 departamentos (la que da el enunciado)
departamentos = ["Amazonas", "Áncash", "Apurímac", "Arequipa", "Ayacucho",
                 "Cajamarca", "Callao", "Cusco", "Huancavelica", "Huánuco",
                 "Ica", "Junín", "La Libertad", "Lambayeque", "Lima", "Loreto",
                 "Madre de Dios", "Moquegua", "Pasco", "Piura", "Puno",
                 "San Martín", "Tacna", "Tumbes", "Ucayali"]

def sin_tildes(texto):
    # NFD separa cada letra de su tilde ("á" → "a" + "´");
    # luego descartamos las tildes (categoría "Mn"). Las MAYÚSCULAS se mantienen.
    return "".join(c for c in unicodedata.normalize("NFD", texto)
                   if unicodedata.category(c) != "Mn")

def departamentos_en(titulo):
    t = sin_tildes(titulo)   # quitamos tildes al título para que "Ancash" o "Junin" también cuenten
    # \b = límite de palabra: "Ica" solo coincide como palabra completa, no dentro de "Huancavelica".
    # Buscamos en el título con mayúsculas, como pide el enunciado.
    # re.search responde sí o no (no cuenta cuántas veces): si "Ica" aparece dos veces
    # ("provincia de Ica del departamento de Ica"), igual se agrega UNA sola vez a la lista.
    return [d for d in departamentos if re.search(r"\b" + sin_tildes(d) + r"\b", t)]

# Una lista de departamentos por decreto, y cuántos son
lluvias["departamentos"] = lluvias["titulo"].apply(departamentos_en)
lluvias["n_departamentos"] = lluvias["departamentos"].str.len()

# Control: ningún decreto debería quedar sin departamentos
print("Decretos sin ningún departamento detectado:", (lluvias["n_departamentos"] == 0).sum())
lluvias[["numero", "tipo", "n_departamentos", "departamentos"]]

Decretos sin ningún departamento detectado: 0


,numero,tipo,n_departamentos,departamentos
0,Decreto Supremo N.° 007-2024-PCM,prorroga,1,[Cusco]
1,Decreto Supremo N.° 006-2024-PCM,prorroga,13,"[Amazonas, Áncash, Arequipa, Ayacucho, Cajamar..."
2,Decreto Supremo N.° 005-2024-PCM,declara,8,"[Arequipa, Ayacucho, Cusco, Huancavelica, Huán..."
3,Decreto Supremo N.° 004-2024-PCM,declara,1,[Apurímac]
4,Decreto Supremo N.° 021-2024-PCM,declara,5,"[Arequipa, Ica, Madre de Dios, Moquegua, Ucayali]"
5,Decreto Supremo N.° 020-2024-PCM,declara,17,"[Amazonas, Áncash, Apurímac, Arequipa, Ayacuch..."
6,Decreto Supremo N.° 016-2024-PCM,declara,15,"[Amazonas, Áncash, Apurímac, Arequipa, Ayacuch..."
7,Decreto Supremo N.° 0012-2024-PCM,declara,7,"[Áncash, Apurímac, Cusco, Huancavelica, Lima, ..."
8,Decreto Supremo N.° 033-2024-PCM,prorroga,8,"[Arequipa, Ayacucho, Cusco, Huancavelica, Huán..."
9,Decreto Supremo N.° 032-2024-PCM,prorroga,7,"[Áncash, Apurímac, Cusco, Huancavelica, Lima, ..."


In [72]:
#Se verifica el conteo
# ---------------------------------------------------------------
# Verificación 1: el caso Ica / Huancavelica
# ---------------------------------------------------------------
# Forma CORRECTA: "Ica" como palabra completa, en el título original
correcta = lluvias["titulo"].apply(lambda t: bool(re.search(r"\bIca\b", t)))
# Forma INCORRECTA: buscar "ica" en el título pasado a minúsculas
incorrecta = lluvias["titulo"].str.lower().str.contains("ica")

print("Decretos que cuentan para Ica:")
print("  con la búsqueda correcta (\\bIca\\b):  ", correcta.sum())
print("  con la búsqueda incorrecta ('ica' in): ", incorrecta.sum())

# Decretos que la forma incorrecta sumaría a Ica por error, y por qué
errores = lluvias[incorrecta & ~correcta]
for _, f in errores.iterrows():
    # palabras del título que contienen "ica" (ej. Huancavelica, pública, hídrica)
    culpables = sorted({p for p in re.findall(r"\w+", f["titulo"]) if "ica" in p.lower()})
    print(f"  ✗ {f['numero']}: se contaría por {culpables}")

# Un "Ica" repetido en el título suma una sola vez
repetidos = lluvias[lluvias["titulo"].apply(lambda t: len(re.findall(r"\bIca\b", t))) > 1]
for _, f in repetidos.iterrows():
    print(f"  ✓ {f['numero']}: 'Ica' aparece varias veces y en la lista está",
          f["departamentos"].count("Ica"), "vez")

# ---------------------------------------------------------------
# Verificación 2: nombres que también son distritos en otro departamento
# ---------------------------------------------------------------
# "San Martín de Porres" (Lima), "San Martín de Pangoa" (Junín), distrito "La Libertad" (Áncash)...
# Mostramos el contexto para confirmar que se mencionan como DEPARTAMENTO.
print("\nContexto de nombres que pueden ser distritos homónimos:")
for _, f in lluvias.iterrows():
    t = sin_tildes(f["titulo"])
    for d in set(f["departamentos"]) & {"Lima", "San Martín", "La Libertad"}:
        for m in re.finditer(r"\b" + sin_tildes(d) + r"\b", t):
            print(f"  {f['numero']} | {d}: ...{t[max(0, m.start() - 35): m.end() + 20]}...")

# ---------------------------------------------------------------
# Resumen: cuántos decretos mencionan cada departamento
# ---------------------------------------------------------------
print("\nDecretos que mencionan cada departamento:")
print(lluvias.explode("departamentos")["departamentos"].value_counts())

Decretos que cuentan para Ica:
  con la búsqueda correcta (\bIca\b):   7
  con la búsqueda incorrecta ('ica' in):  14
  ✗ Decreto Supremo N.° 016-2024-PCM: se contaría por ['Huancavelica']
  ✗ Decreto Supremo N.° 0012-2024-PCM: se contaría por ['Huancavelica']
  ✗ Decreto Supremo N.° 032-2024-PCM: se contaría por ['Huancavelica']
  ✗ Decreto Supremo N.° 027-2024-PCM: se contaría por ['Huancavelica']
  ✗ Decreto Supremo N.° 026-2024-PCM: se contaría por ['Huancavelica']
  ✗ Decreto Supremo N.° 044-2024-PCM: se contaría por ['Huancavelica']
  ✗ Decreto Supremo N.° 039-2024-PCM: se contaría por ['Huancavelica']

Contexto de nombres que pueden ser distritos homónimos:
  Decreto Supremo N.° 006-2024-PCM | San Martín: ...d, Lambayeque, Lima, Pasco, Piura, San Martin y Tumbes, por pelig...
  Decreto Supremo N.° 006-2024-PCM | La Libertad: ...requipa, Ayacucho, Cajamarca, Ica, La Libertad, Lambayeque, Lima, ...
  Decreto Supremo N.° 006-2024-PCM | Lima: ...rca, Ica, La Libertad, Lambayeque, Li